In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv('/content/credit_card_fraud_10k.csv')
df.head()

,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
0,1,84.47,22,Electronics,0,0,66,3,40,0
1,2,541.82,3,Travel,1,0,87,1,64,0
2,3,237.01,17,Grocery,0,0,49,1,61,0
3,4,164.33,4,Grocery,0,1,72,3,34,0
4,5,30.53,15,Food,0,0,79,0,44,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       10000 non-null  int64  
 1   amount               10000 non-null  float64
 2   transaction_hour     10000 non-null  int64  
 3   merchant_category    10000 non-null  object 
 4   foreign_transaction  10000 non-null  int64  
 5   location_mismatch    10000 non-null  int64  
 6   device_trust_score   10000 non-null  int64  
 7   velocity_last_24h    10000 non-null  int64  
 8   cardholder_age       10000 non-null  int64  
 9   is_fraud             10000 non-null  int64  
dtypes: float64(1), int64(8), object(1)
memory usage: 781.4+ KB


In [ ]:
df.describe()

,transaction_id,amount,transaction_hour,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,5000.50000,175.949849,11.593300,0.097800,0.085700,61.798900,2.008900,43.468700,0.015100
std,2886.89568,175.392827,6.922708,0.297059,0.279935,21.487053,1.432559,14.979147,0.121957
min,1.00000,0.000000,0.000000,0.000000,0.000000,25.000000,0.000000,18.000000,0.000000
25%,2500.75000,50.905000,6.000000,0.000000,0.000000,43.000000,1.000000,30.000000,0.000000
50%,5000.50000,122.095000,12.000000,0.000000,0.000000,62.000000,2.000000,44.000000,0.000000
75%,7500.25000,242.480000,18.000000,0.000000,0.000000,80.000000,3.000000,56.000000,0.000000
max,10000.00000,1471.040000,23.000000,1.000000,1.000000,99.000000,9.000000,69.000000,1.000000


In [ ]:
df.isna().sum()

,0
transaction_id,0
amount,0
transaction_hour,0
merchant_category,0
foreign_transaction,0
location_mismatch,0
device_trust_score,0
velocity_last_24h,0
cardholder_age,0
is_fraud,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.dtypes

,0
transaction_id,int64
amount,float64
transaction_hour,int64
merchant_category,object
foreign_transaction,int64
location_mismatch,int64
device_trust_score,int64
velocity_last_24h,int64
cardholder_age,int64
is_fraud,int64


In [ ]:
# Dataset is highly imbalanced (~98.5% normal, ~1.5% fraud).
# Accuracy is misleading — focus on Precision, Recall, F1.
# Use class weights or SMOTE to handle imbalance.
df['is_fraud'].value_counts()
df['is_fraud'].value_counts(normalize=True)

,proportion
is_fraud,
0,0.9849
1,0.0151


In [ ]:
# Transactions above the 95th percentile are rare high-value cases
# Transactions above the 99th percentile are extreme outliers
df['amount'].quantile([0.95, 0.99])

,amount
0.95,530.2085
0.99,796.3233


In [ ]:
df.groupby('is_fraud')['amount'].mean()
df.groupby('is_fraud')['velocity_last_24h'].mean()

,velocity_last_24h
is_fraud,
0,1.990557
1,3.205298


In [ ]:
#EDA fraud vs non-fraud

In [ ]:
#Compare mean amount
df.groupby('is_fraud')['amount'].mean()

,amount
is_fraud,
0,175.333015
1,216.182980


In [ ]:
#Compare velocity
df.groupby('is_fraud')['velocity_last_24h'].mean()

,velocity_last_24h
is_fraud,
0,1.990557
1,3.205298


In [ ]:
df.groupby(['transaction_hour','is_fraud']).size().unstack(fill_value=0)
# Fraud transactions are higher during late night hours (0–2),
# indicating increased risk during midnight period.

is_fraud,0,1
transaction_hour,,
0,380,37
1,391,34
2,362,23
3,398,27
4,383,1
5,417,2
6,411,2
7,425,0
8,421,3


In [ ]:
# Most frequent merchant category for each class (fraud vs normal)
df.groupby('is_fraud')['merchant_category'].agg(lambda x: x.mode()[0])

,merchant_category
is_fraud,
0,Food
1,Grocery


In [ ]:
# EDA for fraud rates
# Overall fraud rate
df['is_fraud'].mean()

np.float64(0.0151)

In [ ]:
df.groupby('merchant_category')['is_fraud'].mean().sort_values(ascending=False)

,is_fraud
merchant_category,
Grocery,0.020062
Food,0.016722
Travel,0.014573
Electronics,0.012480
Clothing,0.011707


In [ ]:
df.groupby('transaction_hour')['is_fraud'].mean().sort_values(ascending=False)

,is_fraud
transaction_hour,
0,0.088729
1,0.080000
3,0.063529
2,0.059740
19,0.009501
23,0.007160
8,0.007075
11,0.005195
6,0.004843


In [ ]:
df.groupby('foreign_transaction')['is_fraud'].mean()

,is_fraud
foreign_transaction,
0,0.007648
1,0.083845


In [ ]:
df.groupby('location_mismatch')['is_fraud'].mean()

,is_fraud
location_mismatch,
0,0.008640
1,0.084014


In [ ]:
df.groupby('velocity_last_24h')['is_fraud'].mean()

,is_fraud
velocity_last_24h,
0,0.010160
1,0.007795
2,0.010518
3,0.011765
4,0.010707
5,0.115068
6,0.083969
7,0.095238
8,0.000000


In [ ]:
# Feature engineering
# High amoount flag (95th percentile)
threshold = df['amount'].quantile(0.95)
df['high_amount_flag'] = (df['amount'] > threshold).astype(int)

In [ ]:
# Night Transaction Flag (0–5 AM)
df['night_transaction_flag'] = (df['transaction_hour'].between(0,5)).astype(int)

In [ ]:
# Low Device Trust Flag (<30)
df['low_trust_flag'] = (df['device_trust_score'] < 30).astype(int)

In [ ]:
df.head(2)

,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud,high_amount_flag,night_transaction_flag,low_trust_flag
0,1,84.47,22,Electronics,0,0,66,3,40,0,0,0,0
1,2,541.82,3,Travel,1,0,87,1,64,0,1,1,0


In [ ]:
# High amoount flag (95th percentile) fraud rate
df.groupby('high_amount_flag')['is_fraud'].mean()

,is_fraud
high_amount_flag,
0,0.014316
1,0.030000


In [ ]:
# Night Transaction Flag (0–5 AM)
df.groupby('night_transaction_flag')['is_fraud'].mean()

,is_fraud
night_transaction_flag,
0,0.003579
1,0.050509


In [ ]:
# Low Device Trust Flag (<30)
df.groupby('low_trust_flag')['is_fraud'].mean()

,is_fraud
low_trust_flag,
0,0.011375
1,0.066079
